# Example 1.1.2 / 1: Defining a Set with =

Companion notebook for **1.1.2 Set Declaration vs. Definition** in the AMPL lab Documentation, section "Defining a Set with `=`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Defining a Set with `=`

With `= sexpr`, AMPL computes the set from the expression. Use it for sets that are part of the
model's structure, or that are derived from other data. In `data.py` and `run.py`, `ampl` is an
`AMPL` object that has already read `model.mod`. Python gives values only to the declared sets and
parameters; the defined sets are read back with `to_list()` like any other set, and
`ampl.param["cost"]["coils"] = 5` changes one value of a parameter:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SHIFTS = {"Day", "Evening", "Night"};       # fixed by the model

param first_year integer;
param last_year integer > first_year;
set YEARS = first_year .. last_year;            # computed from data

set PROD;
param cost {PROD} >= 0;
set CHEAP = {p in PROD: cost[p] < 10};          # derived from other data

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["first_year"] = 2024
ampl.param["last_year"] = 2028
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["cost"] = {"bands": 8, "coils": 12, "plate": 9}

In [5]:
# run.py
print(ampl.set["SHIFTS"].to_list())
print(ampl.set["YEARS"].to_list())
print(ampl.set["CHEAP"].to_list())
ampl.param["cost"]["coils"] = 5
print(ampl.set["CHEAP"].to_list())

['Day', 'Evening', 'Night']
[2024, 2025, 2026, 2027, 2028]
['bands', 'plate']
['bands', 'coils', 'plate']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. In a script, `let` changes a parameter's value:

In [6]:
%%writefile data.dat
param first_year := 2024;
param last_year := 2028;
set PROD := bands coils plate;
param cost := bands 8  coils 12  plate 9;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display SHIFTS, YEARS, CHEAP;
let cost["coils"] := 5;
display CHEAP;

set SHIFTS := Day Evening Night;

set YEARS := 2024 2025 2026 2027 2028;

set CHEAP := bands plate;

set CHEAP := bands coils plate;

